1. 압축 파일 내 아미노산 30~50개 사이의 데이터 추출하는 코드

In [ ]:
import os
import zipfile
import pandas as pd
import csv
import codecs
from google.colab import drive

print("🔗 구글 드라이브 연동을 시작합니다...")
# 이미 마운트되어 있어도 안전하게 통과됩니다.
drive.mount('/content/drive')

drive_path = '/content/drive/MyDrive/Peptide_Project'
save_path = f"{drive_path}/long_peptide_dataset.csv"

files = {
    "T_Cell": "tcell_full_v3.zip",
    "B_Cell": "bcell_full_v3_single_file.zip"
}

combined_df = pd.DataFrame()

print("\n📥 드라이브에서 데이터 추출 및 자동 정제를 시작합니다. (약 1~2분 소요)")

for cell_type, filename in files.items():
    zip_filepath = os.path.join(drive_path, filename)

    if not os.path.exists(zip_filepath):
        print(f"❌ [에러] 드라이브에 {filename} 파일이 없습니다.")
        continue

    print(f"[{cell_type}] 압축 해제 및 이중 헤더(Double Header) 위치 자동 탐지 중...")

    with zipfile.ZipFile(zip_filepath, 'r') as z:
        csv_filename = z.namelist()[0]

        # 1. 원시 텍스트에서 이중 헤더 직접 스캔 (Pandas 오작동 방지)
        with z.open(csv_filename) as f:
            # utf-8-sig를 사용하여 엑셀 특수문자(BOM)까지 깔끔하게 처리
            reader = csv.reader(codecs.iterdecode(f, 'utf-8-sig'))
            headers1 = next(reader) # 첫 번째 줄 (대분류)
            headers2 = next(reader) # 두 번째 줄 (소분류)

    # 2. 정확한 컬럼 인덱스(번호) 찾기
    seq_idx = -1
    assay_idx = -1

    for i in range(len(headers1)):
        h1 = headers1[i].strip()
        h2 = headers2[i].strip()

        # Epitope 그룹의 Name (펩타이드 서열)
        if 'Epitope' in h1 and h2 in ['Name', 'Description']:
            seq_idx = i
        # Assay 그룹의 Qualitative Measurement (정답 라벨)
        elif 'Assay' in h1 and 'Qualitative' in h2:
            assay_idx = i

    if seq_idx == -1 or assay_idx == -1:
        print(f"❌ [{cell_type}] 컬럼 위치 탐지 실패. (서열: {seq_idx}, 정답: {assay_idx})")
        continue

    print(f"[{cell_type}] 컬럼 위치 탐지 완료! (서열 열번호: {seq_idx}, 정답 열번호: {assay_idx})")

    # 3. 데이터 로드 (탐지된 인덱스만 뽑아서 알맹이만 읽기)
    print(f"[{cell_type}] 데이터 로드 및 30~50 길이 필터링 중...")
    with zipfile.ZipFile(zip_filepath, 'r') as z:
        z.extract(csv_filename, '/content/')

    csv_path = f"/content/{csv_filename}"

    try:
        # 핵심: skiprows=2를 통해 복잡한 문자열 헤더를 모두 날려버리고 순수 데이터만 추출
        df = pd.read_csv(csv_path, usecols=[seq_idx, assay_idx], skiprows=2, header=None, low_memory=False, on_bad_lines='skip')
    except Exception as e:
        print(f"[{cell_type}] CSV 읽기 오류: {e}")
        os.remove(csv_path)
        continue

    # 추출한 데이터의 열 순서에 맞게 컬럼명 수동 지정
    if seq_idx < assay_idx:
        df.columns = ['Sequence', 'Assay_Result']
    else:
        df.columns = ['Assay_Result', 'Sequence']

    # 결측치 제거 및 순수 알파벳 서열만 추출
    df = df.dropna()
    df['Sequence'] = df['Sequence'].astype(str)
    df = df[df['Sequence'].str.isalpha()]

    # 🔥 30~50 길이 필터링
    df['Seq_Length'] = df['Sequence'].apply(len)
    df_filtered = df[(df['Seq_Length'] >= 30) & (df['Seq_Length'] <= 50)].copy()

    combined_df = pd.concat([combined_df, df_filtered], ignore_index=True)
    os.remove(csv_path) # 메모리 확보

if combined_df.empty:
    print("❌ 추출된 데이터가 없습니다. 과정을 중단합니다.")
else:
    # 4. 정답 라벨(Target) 이진화
    print("\n⚙️ 데이터 라벨링(Positive=1, Negative=0) 및 중복 서열 제거 중...")
    def map_label(val):
        val = str(val).lower()
        if 'positive' in val: return 1
        elif 'negative' in val: return 0
        else: return None

    combined_df['Target'] = combined_df['Assay_Result'].apply(map_label)
    combined_df = combined_df.dropna(subset=['Target'])

    # 중복 서열 제거
    final_df = combined_df[['Sequence', 'Target', 'Seq_Length']].drop_duplicates(subset=['Sequence'])
    final_df['Target'] = final_df['Target'].astype(int)

    # 5. 최종 결과를 구글 드라이브에 안전하게 저장
    final_df.to_csv(save_path, index=False)

    print("\n=== 🎉 완벽 자동 수집 및 정제 완료 ===")
    print(f" - 확보된 30~50 길이의 고유 서열 개수: {len(final_df):,} 개")
    print(f" - 양성(염증/이물 반응 O, Label 1) 데이터: {len(final_df[final_df['Target'] == 1]):,} 개")
    print(f" - 음성(안전함, Label 0) 데이터: {len(final_df[final_df['Target'] == 0]):,} 개")
    print(f" - 💾 최종 저장 완료: 드라이브의 '창종설_데이터/long_peptide_dataset.csv' 에 저장됨.")

🔗 구글 드라이브 연동을 시작합니다...
Mounted at /content/drive

📥 드라이브에서 데이터 추출 및 자동 정제를 시작합니다. (약 1~2분 소요)
[T_Cell] 압축 해제 및 이중 헤더(Double Header) 위치 자동 탐지 중...
[T_Cell] 컬럼 위치 탐지 완료! (서열 열번호: 11, 정답 열번호: 122)
[T_Cell] 데이터 로드 및 30~50 길이 필터링 중...
[B_Cell] 압축 해제 및 이중 헤더(Double Header) 위치 자동 탐지 중...
[B_Cell] 컬럼 위치 탐지 완료! (서열 열번호: 11, 정답 열번호: 102)
[B_Cell] 데이터 로드 및 30~50 길이 필터링 중...

⚙️ 데이터 라벨링(Positive=1, Negative=0) 및 중복 서열 제거 중...

=== 🎉 완벽 자동 수집 및 정제 완료 ===
 - 확보된 30~50 길이의 고유 서열 개수: 9,847 개
 - 양성(염증/이물 반응 O, Label 1) 데이터: 5,013 개
 - 음성(안전함, Label 0) 데이터: 4,834 개
 - 💾 최종 저장 완료: 드라이브의 '창종설_데이터/long_peptide_dataset.csv' 에 저장됨.


2. 파일 내 아미노산 5~50개 사이의 데이터 추출 및 전처리하는 코드 (시스템 코드에서 이용할 데이터입니다.)

In [ ]:
import os
import zipfile
import pandas as pd
import csv
import codecs
from google.colab import drive

# 0. 구글 드라이브 강제 재연결 (Transport endpoint 에러 방지)
print("🔄 구글 드라이브 연결을 재설정합니다...")
drive.mount('/content/drive', force_remount=True)

drive_path = '/content/drive/MyDrive/Peptide_Project'
# 5~50-mer 전용 파일로 새롭게 명명하여 저장합니다.
save_path = f"{drive_path}/peptide_dataset_5_50.csv"

files = {
    "T_Cell": "tcell_full_v3.zip",
    "B_Cell": "bcell_full_v3_single_file.zip"
}

combined_df = pd.DataFrame()

print("\n📥 드라이브에서 데이터 추출 및 자동 정제를 시작합니다. (5~50mer 타겟팅)")

for cell_type, filename in files.items():
    zip_filepath = os.path.join(drive_path, filename)

    if not os.path.exists(zip_filepath):
        print(f"❌ [에러] 드라이브에 {filename} 파일이 없습니다.")
        continue

    print(f"[{cell_type}] 압축 해제 및 이중 헤더(Double Header) 위치 자동 탐지 중...")

    with zipfile.ZipFile(zip_filepath, 'r') as z:
        csv_filename = z.namelist()[0]

        # 1. 원시 텍스트에서 이중 헤더 직접 스캔
        with z.open(csv_filename) as f:
            reader = csv.reader(codecs.iterdecode(f, 'utf-8-sig'))
            headers1 = next(reader) # 첫 번째 줄 (대분류)
            headers2 = next(reader) # 두 번째 줄 (소분류)

    # 2. 정확한 컬럼 인덱스(번호) 찾기
    seq_idx = -1
    assay_idx = -1

    for i in range(len(headers1)):
        h1 = headers1[i].strip()
        h2 = headers2[i].strip()

        if 'Epitope' in h1 and h2 in ['Name', 'Description']:
            seq_idx = i
        elif 'Assay' in h1 and 'Qualitative' in h2:
            assay_idx = i

    if seq_idx == -1 or assay_idx == -1:
        print(f"❌ [{cell_type}] 컬럼 위치 탐지 실패.")
        continue

    print(f"[{cell_type}] 컬럼 위치 탐지 완료! (서열: {seq_idx}, 정답: {assay_idx})")

    # 3. 데이터 로드
    print(f"[{cell_type}] 데이터 로드 및 5~50 길이 필터링 중...")
    with zipfile.ZipFile(zip_filepath, 'r') as z:
        z.extract(csv_filename, '/content/')

    csv_path = f"/content/{csv_filename}"

    try:
        df = pd.read_csv(csv_path, usecols=[seq_idx, assay_idx], skiprows=2, header=None, low_memory=False, on_bad_lines='skip')
    except Exception as e:
        print(f"[{cell_type}] CSV 읽기 오류: {e}")
        os.remove(csv_path)
        continue

    if seq_idx < assay_idx:
        df.columns = ['Sequence', 'Assay_Result']
    else:
        df.columns = ['Assay_Result', 'Sequence']

    # 결측치 제거 및 순수 알파벳 서열만 추출
    df = df.dropna()
    df['Sequence'] = df['Sequence'].astype(str)
    df = df[df['Sequence'].str.isalpha()]

    # 🔥 [수정됨] 5~50 길이 필터링 적용! (초단기 모티프부터 3차원 접힘까지 커버)
    df['Seq_Length'] = df['Sequence'].apply(len)
    df_filtered = df[(df['Seq_Length'] >= 5) & (df['Seq_Length'] <= 50)].copy()

    combined_df = pd.concat([combined_df, df_filtered], ignore_index=True)
    os.remove(csv_path)

if combined_df.empty:
    print("❌ 추출된 데이터가 없습니다.")
else:
    # 4. 정답 라벨 이진화 및 중복 제거
    print("\n⚙️ 데이터 라벨링(Positive=1, Negative=0) 및 중복 서열 제거 중...")
    def map_label(val):
        val = str(val).lower()
        if 'positive' in val: return 1
        elif 'negative' in val: return 0
        else: return None

    combined_df['Target'] = combined_df['Assay_Result'].apply(map_label)
    combined_df = combined_df.dropna(subset=['Target'])

    final_df = combined_df[['Sequence', 'Target', 'Seq_Length']].drop_duplicates(subset=['Sequence'])
    final_df['Target'] = final_df['Target'].astype(int)

    # 5. 구글 드라이브에 최종 저장
    final_df.to_csv(save_path, index=False)

    print("\n=== 🎉 5~50-mer 데이터 구축 완료 ===")
    print(f" - 확보된 전체 서열 개수: {len(final_df):,} 개")
    print(f" - 양성(염증 유발, 1) 데이터: {len(final_df[final_df['Target'] == 1]):,} 개")
    print(f" - 음성(안전, 0) 데이터: {len(final_df[final_df['Target'] == 0]):,} 개")
    print(f" - 💾 파일 저장 완료: {save_path}")

🔄 구글 드라이브 연결을 재설정합니다...
Mounted at /content/drive

📥 드라이브에서 데이터 추출 및 자동 정제를 시작합니다. (5~50mer 타겟팅)
[T_Cell] 압축 해제 및 이중 헤더(Double Header) 위치 자동 탐지 중...
[T_Cell] 컬럼 위치 탐지 완료! (서열: 11, 정답: 122)
[T_Cell] 데이터 로드 및 5~50 길이 필터링 중...
[B_Cell] 압축 해제 및 이중 헤더(Double Header) 위치 자동 탐지 중...
[B_Cell] 컬럼 위치 탐지 완료! (서열: 11, 정답: 102)
[B_Cell] 데이터 로드 및 5~50 길이 필터링 중...

⚙️ 데이터 라벨링(Positive=1, Negative=0) 및 중복 서열 제거 중...

=== 🎉 5~50-mer 데이터 구축 완료 ===
 - 확보된 전체 서열 개수: 803,648 개
 - 양성(염증 유발, 1) 데이터: 226,403 개
 - 음성(안전, 0) 데이터: 577,245 개
 - 💾 파일 저장 완료: /content/drive/MyDrive/Peptide_Project/peptide_dataset_5_50.csv
